# Stage 3, step 4: LucaOne's input limit, and IsoFormer's aggregator on GTEx

Two items from Mina's research plan, Priority 2. Gantt 41 asks for LucaOne as a frozen
baseline on mRNA stability and GTEx. Gantt 44, our addition, asks for IsoFormer's aggregator
on the GTEx transcript-expression pilot.

They are very different jobs. LucaOne's is a feasibility question and the answer turns out to
be mostly no, for a reason worth knowing before any compute is spent. IsoFormer's is a build
and a check, and it matters more than its place on the list suggests, because **IsoFormer
solves the alignment problem by not having one**. Where BioLangFusion resamples all three
streams onto the codon frame and combines them position by position, IsoFormer leaves every
modality at its own length and width and lets cross-attention find the correspondence. Step 3
showed BioLangFusion's codon alignment was silently two codons out of register and that the
only head unharmed by it was the one that attends to context rather than to position. An
architecture that refuses to align at all is the right thing to put next to that.

The model is in [`../shared_code/fusion/isoformer.py`](../shared_code/fusion/isoformer.py),
checked by `tests/test_isoformer.py` on small tensors.

---

In [1]:
# Standard library
import json
import os
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage3
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import datasets, encoders
from shared_code.fusion import isoformer, train
from shared_code.fusion import token_embeddings as tokens

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| Apple GPU available:", torch.backends.mps.is_available())

torch 2.12.1 | Apple GPU available: True


## Part 1: LucaOne, and whether it can read our data

LucaOne is the alternative this project deliberately does not take: one 1.8B-parameter model
trained from scratch on a single 39-token vocabulary spanning nucleotides and amino acids,
so cross-modal structure is learned in pretraining rather than fused afterwards. Julian's
review rates it Priority Low for exactly that reason, and its value here is as a reference
point.

Before any of that, one number decides what is possible: **LucaOne's maximum input is 1,280
tokens**, and its tokenization is character-level, one token per nucleotide or per residue
with no codon grouping. So a coding sequence costs its full length in tokens, and the mode
that makes LucaOne interesting, nucleotides and protein in the same sequence, costs the
length plus a third of it again.

In [2]:
LUCAONE_LIMIT = 1280          # Methods, "Vocabulary": max length 1,280 tokens
SPECIALS = 2                  # [CLS] and [SEP]

gtex = pd.read_csv(datasets.DEFAULT_DATA_DIR / isoformer.GTEX_PILOT)
rows = []
for column in ("DNA", "RNA", "5UTR", "CDS", "3UTR", "Protein"):
    length = gtex[column].astype(str).str.strip().str.len()
    rows.append({"GTEx column": column, "min": length.min(), "median": int(length.median()),
                 "max": length.max(),
                 "fits LucaOne": f"{(length + SPECIALS <= LUCAONE_LIMIT).mean() * 100:.1f}%"})
display(pd.DataFrame(rows).set_index("GTEx column"))

benchmarks = []
for key in datasets.BLF_BENCHMARKS:
    length = datasets.blf_benchmark_rows(key)["length"]
    benchmarks.append({
        "benchmark": key, "rows": len(length), "max nt": length.max(),
        "nucleotides alone": f"{(length + SPECIALS <= LUCAONE_LIMIT).mean() * 100:.1f}%",
        "protein alone": f"{(length // 3 + SPECIALS <= LUCAONE_LIMIT).mean() * 100:.1f}%",
        "both in one sequence": f"{(length + length // 3 + SPECIALS + 1 <= LUCAONE_LIMIT).mean() * 100:.1f}%"})
display(pd.DataFrame(benchmarks).set_index("benchmark"))
print(f"The joint mode needs 4L/3 + {SPECIALS + 1} <= {LUCAONE_LIMIT}, "
      f"so it caps the coding sequence at {(LUCAONE_LIMIT - SPECIALS - 1) * 3 // 4} nt.")

,min,median,max,fits LucaOne
GTEx column,,,,
DNA,6000,6000,6000,0.0%
RNA,78,1723,21116,39.9%
5UTR,1,152,3318,99.6%
CDS,3,760,16080,68.1%
3UTR,1,390,17986,76.6%
Protein,1,253,5360,96.8%


,rows,max nt,nucleotides alone,protein alone,both in one sequence
benchmark,,,,,
fungal_expression,3138,999,100.0%,100.0%,93.7%
ecoli_proteins,4450,999,100.0%,100.0%,96.4%
mrna_stability,23929,999,100.0%,100.0%,93.4%
cov_vac_degradation,2400,81,100.0%,100.0%,100.0%


The joint mode needs 4L/3 + 3 <= 1280, so it caps the coding sequence at 957 nt.


That settles most of Gantt 41 before a single embedding is computed.

**On GTEx, LucaOne cannot do the job.** The pilot's DNA track is a fixed 6,000-nucleotide
genomic window, which is almost five times the context, so none of it fits. Fewer than half
the transcripts fit either. A LucaOne GTEx baseline would therefore have to drop the DNA
modality entirely and discard most of the RNA, which is not a baseline for a multimodal
comparison, it is a different experiment.

**On the BLF benchmarks it works for one stream at a time**, because the 999-nucleotide
preprocessing rule leaves room. Its distinctive joint mode does not quite fit: 4L/3 tokens
caps the coding sequence at 957 nucleotides and loses a few percent of fungal and stability.

So the honest scope is a LucaOne frozen baseline on mRNA stability, single stream, and
nothing on GTEx. Whether even that is worth 1.8B parameters of inference on a laptop is a
question for Mina; the review already rates the paper's relevance low.

The second question is whether usable weights exist at all. The cell below asks the hub
rather than assuming, and downloads nothing. The answer turns out to be yes: the group
publishes a joint checkpoint as well as the gene-only and protein-only ablations, with
`config.json` and sharded safetensors. So availability is not the obstacle. The context
is.

In [3]:
# Does a loadable LucaOne checkpoint exist? Metadata only, no weights are fetched.
found = []
try:
    from huggingface_hub import HfApi
    api = HfApi()
    # Ask the hub what the group actually published rather than guessing repository names.
    published = [m.id for m in api.list_models(author="LucaGroup")]
    print(f"LucaGroup publishes {len(published)} models:", ", ".join(sorted(published)))
    CANDIDATES = sorted(m for m in published if "lucaone" in m.lower()) or published[:4]
    for repo in CANDIDATES:
        try:
            info = api.model_info(repo, files_metadata=False)
            files = [s.rfilename for s in info.siblings] if info.siblings else []
            found.append({"repo": repo, "exists": "yes",
                          "config.json": "yes" if "config.json" in files else "no",
                          "files": len(files),
                          "weights": ", ".join(f for f in files
                                               if f.endswith((".bin", ".safetensors", ".pt")))[:60]})
        except Exception as problem:
            found.append({"repo": repo, "exists": f"no ({type(problem).__name__})",
                          "config.json": "", "files": 0, "weights": ""})
except ImportError:
    print("huggingface_hub is not installed; skipping the probe")

if found:
    display(pd.DataFrame(found).set_index("repo"))
    print("A repo without config.json will not load through transformers AutoModel and\n"
          "needs LucaOne's own loader from its GitHub release.")

LucaGroup publishes 13 models: LucaGroup/LucaCell-v1.0-step6.4M, LucaGroup/LucaOne-LucaCell-step60M, LucaGroup/LucaOne-default-step17.6M, LucaGroup/LucaOne-default-step36M, LucaGroup/LucaOne-default-step5.6M, LucaGroup/LucaOne-gene-step36.8M, LucaGroup/LucaOne-mask-step36M, LucaGroup/LucaOne-prot-step38.2M, LucaGroup/LucaVirus-default-step3.8M, LucaGroup/LucaVirus-gene-step3.8M, LucaGroup/LucaVirus-mask-step1.4M, LucaGroup/LucaVirus-mask-step3.8M, LucaGroup/LucaVirus-prot-step3.8M


,exists,config.json,files,weights
repo,,,,
LucaGroup/LucaOne-LucaCell-step60M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-default-step17.6M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-default-step36M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-default-step5.6M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-gene-step36.8M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-mask-step36M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."
LucaGroup/LucaOne-prot-step38.2M,yes,yes,13,"model-00001-of-00002.safetensors, model-00002-..."


A repo without config.json will not load through transformers AutoModel and
needs LucaOne's own loader from its GitHub release.


**Verdict on Gantt 41.** LucaOne is out for GTEx on input length alone, and that is a finding
rather than a failure: it is the concrete cost of a character-level vocabulary with a 1,280
token context, and it is worth one line in the write-up next to the claim that joint
pretraining removes the alignment problem. It removes alignment and replaces it with a
context ceiling.

What remains possible, a single-stream LucaOne baseline on mRNA stability, is cheap to
describe and expensive to run, and should be a decision made with Mina rather than a default.

---

## Part 2: IsoFormer's aggregator on the GTEx pilot

In [4]:
# IsoFormer Table 2: the modality ablation, NT as both DNA and RNA encoder, ESM-2 150M for
# protein. R2 and Spearman across tissues, mean and standard deviation over 5 seeds.
ISOFORMER_TABLE2 = pd.DataFrame(
    [("dna",           0.13, 0.02, 0.43, 0.01),
     ("rna",           0.36, 0.03, 0.61, 0.01),
     ("prot",          0.20, 0.01, 0.46, 0.01),
     ("dna+prot",      0.28, 0.01, 0.52, 0.01),
     ("dna+rna",       0.39, 0.01, 0.64, 0.01),
     ("dna+rna+prot",  0.43, 0.01, 0.65, 0.01)],
    columns=["variant", "R2", "R2 sd", "Spearman", "Spearman sd"]).set_index("variant")
display(ISOFORMER_TABLE2)
best_single = ISOFORMER_TABLE2.loc[["dna", "rna", "prot"], "Spearman"].max()
gain = ISOFORMER_TABLE2.loc["dna+rna+prot", "Spearman"] - best_single
print(f"their fusion gain in Spearman: {gain:+.2f}, against a seed sd of "
      f"{ISOFORMER_TABLE2['Spearman sd'].max():.2f}")

,R2,R2 sd,Spearman,Spearman sd
variant,,,,
dna,0.13,0.02,0.43,0.01
rna,0.36,0.03,0.61,0.01
prot,0.20,0.01,0.46,0.01
dna+prot,0.28,0.01,0.52,0.01
dna+rna,0.39,0.01,0.64,0.01
dna+rna+prot,0.43,0.01,0.65,0.01


their fusion gain in Spearman: +0.04, against a seed sd of 0.01


Read that last line against step 3. IsoFormer reports five seeds with a standard deviation
on every number, and its three-modality gain over the best single encoder is several times
that spread. BioLangFusion reports one number per cell with no error bar, and step 3 measured
its headline gain to be the same size as a difference its benchmark cannot resolve. Same
field, one year apart, two very different evidential standards. That comparison belongs in
the write-up on its own.

**The deviation that dominates everything below: IsoFormer fine-tunes its encoders and we
freeze ours.** The paper is explicit, analyzing "the finetuning run in which only the RNA
encoder is used" and concluding that "RNA encoder specialization during finetuning is key to
achieve a strong representation". Our encoders are frozen and read from the step 2 caches,
because frozen fusion is this project's premise. So the gap between our numbers and theirs
is not a reproduction failure, it is a measurement of what freezing costs, which is a
question this project exists to ask. Their DNA-only model reaches 0.43 Spearman; a frozen
DNA encoder with a linear head on top reaches far less, and the distance between those is
the price of not fine-tuning.

**What we can and cannot match.** The pilot is a 2,000-row sample of InstaDeepAI's own
`multi_omics_transcript_expression` release, which is IsoFormer's dataset, so the label
scaling and the thirty tissues are theirs and Stage 1 split it by gene with none shared. What
differs is size: 1,000 training rows against their full set, and our Nucleotide Transformer
is the 100M checkpoint at 512 wide rather than their 250M at 768. Our ESM-2 150M is exactly
theirs.

So the absolute values are not the target. **The ordering is**: more modalities should beat
fewer, RNA should be the strongest single modality, and DNA with RNA should beat DNA with
protein. That is the claim their Table 2 makes, and it is the one a 1,000-row pilot can test.

### Settings

In [5]:
BENCHMARK = "gtex_pilot"
CACHE_ROOT = Path("embeddings")
RESULTS_DIR = Path("results") / BENCHMARK

# IsoFormer uses a fixed batch of 64 (Section 4.3). A fixed batch is what fails here, and
# the reason is padding rather than the model. The DNA window is a fixed 1,000 tokens while
# a transcript runs from 78 to 2,000 and a protein from 1 to 1,022, so a row costs between
# 10 and 81 million attention cells; a fixed batch pads to its longest member and pays that
# worst case for every row in it. A random 16 reached 24 GB on the GPU where a median batch
# needs under one.
#
# So batches are formed under a budget on attention cells instead: many short transcripts
# together, few long ones, and roughly constant memory either way. MAX_BATCH is a ceiling,
# not a target. At this budget the worst batch holds about 1.2 GB of attention maps.
MAX_BATCH = 16
CELL_BUDGET = 300_000_000
VAL_FRACTION = 0.05        # "early stopping on a validation set comprised of 5% of the train set"
MAX_EPOCHS = 500
N_HEADS = 8                # Appendix B, "Cross-Attn: number of heads"
SEEDS = (0,)               # add 1 and 2 once the first pass looks right

# IsoFormer left-crops RNA to 12 kb "to conserve the 3'UTR regions". ESM-2 takes 1,022
# residues. The DNA window is a fixed 6,000 nt and needs no crop.
CROP = {"dna": None, "rna": 12_000, "prot": 1_022}
# The crops must fit the encoders' real context, checked below rather than assumed.
# Nucleotide Transformer v2 100M reports max_position_embeddings 2,050 and ESM-2 150M
# reports 1,026, so 2,048 and 1,022 content tokens. 12,000 nt is 2,000 six-mers, and the
# k-mer remainder adds at most five more; the 6,000 nt DNA window is 1,000 tokens.
CONTEXT = {"ntv2_100m": 2_048, "esm2_150m": 1_022}

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")
VARIANTS = ["dna", "rna", "prot", "dna+prot", "dna+rna", "dna+rna+prot"]

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"{len(gtex):,} transcripts | splits {dict(gtex['split'].value_counts())} | "
      f"{len(datasets.GTEX_TISSUES)} tissues | device {DEVICE}")
print("encoder roles:", {m: isoformer.STREAM_ENCODER[m] for m in isoformer.MODALITIES})

2,000 transcripts | splits {'train': 1000, 'test': 1000} | 30 tissues | device mps
encoder roles: {'dna': 'ntv2_100m', 'rna': 'ntv2_100m', 'prot': 'esm2_150m'}


### Embedding the pilot

Three streams, two of them read by the same encoder, which is why each cache is named for the
stream rather than the encoder. Resumable in the same way as step 2: rerunning skips shards
that already exist.

In [6]:
# Build the input strings, recording how many rows each crop touches.
def stream_texts(modality):
    column = isoformer.STREAM_COLUMN[modality]
    encoder = encoders.ENCODERS[isoformer.STREAM_ENCODER[modality]]
    raw = gtex[column].astype(str).str.strip().str.upper()
    limit = CROP[modality]
    if limit is None:
        kept = raw
    elif modality == "rna":
        kept = raw.str[-limit:]        # left-crop, keeping the 3' end as IsoFormer does
    else:
        kept = raw.str[:limit]
    cropped = int((raw.str.len() > (limit or 10 ** 9)).sum())
    texts = [encoders.protein_input(encoder, s) if encoder.modality == "protein"
             else encoders.encoder_input(encoder, s) for s in kept]
    return texts, cropped


caches = {}
for modality in isoformer.MODALITIES:
    texts, cropped = stream_texts(modality)
    encoder_key = isoformer.STREAM_ENCODER[modality]
    name = f"{modality}_{encoder_key}"
    # token_embeddings raises rather than truncating, so confirm the crop fits first.
    longest = max(len(t.replace(" ", "")) for t in texts)
    worst = (longest if encoders.ENCODERS[encoder_key].modality == "protein"
             else longest // 6 + longest % 6)
    if worst > CONTEXT[encoder_key]:
        raise ValueError(f"{name}: longest input is {worst:,} tokens, over the "
                         f"{CONTEXT[encoder_key]:,} the encoder accepts; lower CROP")
    print(f"  longest input {worst:,} tokens against a {CONTEXT[encoder_key]:,} budget")
    print(f"\n{name}: {len(texts):,} rows, {cropped:,} cropped to {CROP[modality]}")
    cache, computed = tokens.extract(encoders.ENCODERS[encoder_key], texts,
                                     CACHE_ROOT / BENCHMARK / name, DEVICE)
    caches[modality] = cache
    print(f"  {'extracted' if computed else 'already complete'}: {len(cache):,} rows, "
          f"width {cache.width}, {cache.meta['total_tokens']:,} tokens")

  longest input 1,000 tokens against a 2,048 budget

dna_ntv2_100m: 2,000 rows, 0 cropped to None
  already complete: 2,000 rows, width 512, 2,000,000 tokens
  longest input 2,000 tokens against a 2,048 budget

rna_ntv2_100m: 2,000 rows, 11 cropped to 12000
  already complete: 2,000 rows, width 512, 776,882 tokens
  longest input 1,022 tokens against a 1,022 budget

prot_esm2_150m: 2,000 rows, 124 cropped to 1022
  already complete: 2,000 rows, width 640, 696,364 tokens


### The modality ablation

Six models, one per row of their Table 2, through one architecture. A single-modality model
has nothing to attend to and reduces to that stream pooled and read by the linear head, which
is what their single-modality rows are.

In [7]:
STREAMS = isoformer.GTExStreams.open(CACHE_ROOT)
LOADERS = isoformer.loaders_for(STREAMS, batch_size=MAX_BATCH, val_fraction=VAL_FRACTION,
                                budget=CELL_BUDGET, n_heads=N_HEADS)
for name, made in LOADERS.items():
    sizes = [len(b) for b in made.batch_sampler.batches]
    print(f"{name:<6} {len(made.dataset):>5} rows in {len(sizes):>3} batches, "
          f"size {min(sizes)} to {max(sizes)}")
print("widths", STREAMS.widths)

# One forward and backward of the heaviest variant before the sweep commits to the batch
# size. Attention over these lengths is the whole memory cost, so a failure here means
# lowering BATCH_SIZE, not anything else.
# The probe must run the WORST batch, not the first one: the first attempt at this notebook
# passed on batch one and then ran out of memory partway through the sweep.
import time
costs = []
for position, batch in enumerate(LOADERS["train"].batch_sampler.batches):
    picked = LOADERS["train"].dataset.indices[batch]
    longest = {m: int(np.asarray(STREAMS.caches[m].lengths)[picked].max())
               for m in isoformer.MODALITIES}
    costs.append((len(batch) * N_HEADS * sum(longest[a] * longest[b] for a in longest
                                             for b in longest if a != b), position, longest))
worst_cost, worst_position, worst_lengths = max(costs)
print(f"heaviest of {len(costs)} training batches: {worst_lengths}, "
      f"{worst_cost / 1e6:,.0f}M attention cells, {worst_cost * 4 / 1e9:.2f} GB of maps")

probe = isoformer.builder(STREAMS, "dna+rna+prot", None, n_heads=N_HEADS).to(DEVICE)
heaviest = isoformer.collate([LOADERS["train"].dataset[i]
                              for i in LOADERS["train"].batch_sampler.batches[worst_position]])
try:
    started = time.perf_counter()
    out, _ = probe(heaviest.to(DEVICE))
    torch.nn.MSELoss()(out, heaviest.labels.to(DEVICE)).backward()
    seconds = time.perf_counter() - started
    print(f"the heaviest batch fits: {seconds:.1f}s, so roughly "
          f"{seconds * len(costs) / 60:.1f} min per epoch for the three-modality model")
except (RuntimeError, MemoryError) as problem:
    print(f"the heaviest batch does NOT fit: {type(problem).__name__}: {str(problem)[:140]}")
    print("lower CELL_BUDGET and rerun from the loader cell before starting the sweep")
    raise
finally:
    del probe, heaviest
    if DEVICE == "mps":
        torch.mps.empty_cache()

val       50 rows in   7 batches, size 1 to 16
train    950 rows in  91 batches, size 2 to 16
test    1000 rows in  96 batches, size 3 to 16
widths {'dna': 512, 'rna': 512, 'prot': 640}
heaviest of 91 training batches: {'dna': 1000, 'rna': 947, 'prot': 578}, 298M attention cells, 1.19 GB of maps
the heaviest batch fits: 2.0s, so roughly 3.1 min per epoch for the three-modality model


In [8]:
# The sweep. Only start this once the probe above reports that the heaviest batch fits.
for seed in SEEDS:
    print(f"\n--- seed {seed} ---")
    for variant in VARIANTS:
        made = isoformer.loaders_for(STREAMS, batch_size=MAX_BATCH,
                                     val_fraction=VAL_FRACTION, seed=seed,
                                     budget=CELL_BUDGET, n_heads=N_HEADS)
        train.run(STREAMS, variant,
                  train.TrainConfig(batch_size=MAX_BATCH, max_epochs=MAX_EPOCHS, seed=seed),
                  DEVICE, RESULTS_DIR, builder=isoformer.builder, loaders=made,
                  tag=f"{BENCHMARK}__{variant}__heads{N_HEADS}"
                      f"__budget{CELL_BUDGET // 1_000_000}M__seed{seed}",
                  n_heads=N_HEADS)


--- seed 0 ---
  gtex_pilot__dna__heads8__budget300M__seed0: training
    epoch   0  train loss 11.9132  val 0.0171  best 0.0171 @ 0
    epoch  10  train loss 5.8640  val 0.1014  best 0.1014 @ 10
    epoch  20  train loss 5.4083  val 0.1206  best 0.1206 @ 20
    epoch  30  train loss 5.5036  val 0.1406  best 0.1406 @ 30
    epoch  40  train loss 5.5117  val 0.1600  best 0.1600 @ 40
    epoch  50  train loss 5.4998  val 0.1776  best 0.1776 @ 50
    epoch  60  train loss 5.4868  val 0.1912  best 0.1912 @ 60
    epoch  70  train loss 5.4685  val 0.2040  best 0.2040 @ 70
    epoch  80  train loss 5.4463  val 0.2146  best 0.2146 @ 80
    epoch  90  train loss 5.4428  val 0.2214  best 0.2214 @ 90
    epoch 100  train loss 5.4262  val 0.2279  best 0.2279 @ 100
    epoch 110  train loss 5.4140  val 0.2297  best 0.2304 @ 109
    epoch 120  train loss 5.4038  val 0.2316  best 0.2316 @ 120
    epoch 130  train loss 5.3913  val 0.2337  best 0.2337 @ 130
    epoch 140  train loss 5.3871  val 0.232

In [9]:
# Ours beside theirs. Spearman is the monitored score; R2 is recomputed from the stored
# per-row predictions, which is why they are saved.
found = []
for variant in VARIANTS:
    scores, r2, epochs = [], [], []
    for seed in SEEDS:
        path = (RESULTS_DIR / f"{BENCHMARK}__{variant}__heads{N_HEADS}"
                f"__budget{CELL_BUDGET // 1_000_000}M__seed{seed}.json")
        if not path.exists():
            continue
        result = train.Result.load(path)
        scores.append(result.test_score)
        epochs.append(result.epochs)
        r2.append(train.r_squared(np.asarray(result.test_truth),
                                  np.asarray(result.test_prediction)))
    if scores:
        found.append({"variant": variant, "seeds": len(scores),
                      "our Spearman": np.mean(scores), "our R2": np.mean(r2),
                      "their Spearman": ISOFORMER_TABLE2.loc[variant, "Spearman"],
                      "their R2": ISOFORMER_TABLE2.loc[variant, "R2"],
                      "epochs": int(np.mean(epochs)),
                      "hit the cap": "yes" if max(epochs) >= MAX_EPOCHS else ""})
comparison = pd.DataFrame(found).set_index("variant")
display(comparison.round(3))
if (comparison["hit the cap"] == "yes").any():
    capped = list(comparison.index[comparison["hit the cap"] == "yes"])
    print(f"NOTE: {', '.join(capped)} reached MAX_EPOCHS without early stopping, so they were "
          "still improving when training ended.\nA baseline cut short that way makes any gain "
          "measured over it an upper bound, which matters most\nwhen the capped run is one of "
          "the single modalities.")

# The three claims Table 2 makes, which a 1,000-row pilot can test even when the absolute
# values cannot match.
ours = comparison["our Spearman"].to_dict()
singles = [v for v in ("dna", "rna", "prot") if v in ours]
checks = []
if len(singles) == 3:
    checks.append({"claim": "RNA is the strongest single modality",
                   "result": "ok" if max(singles, key=lambda v: ours[v]) == "rna" else "NO",
                   "detail": ", ".join(f"{v} {ours[v]:.3f}" for v in singles)})
if "dna+rna+prot" in ours and singles:
    best = max(ours[v] for v in singles)
    checks.append({"claim": "all three modalities beat the best single one",
                   "result": "ok" if ours["dna+rna+prot"] > best else "NO",
                   "detail": f"three {ours['dna+rna+prot']:.3f} against best single {best:.3f}"})
if "dna+rna" in ours and "dna+prot" in ours:
    checks.append({"claim": "DNA with RNA beats DNA with protein",
                   "result": "ok" if ours["dna+rna"] > ours["dna+prot"] else "NO",
                   "detail": f"dna+rna {ours['dna+rna']:.3f} against "
                             f"dna+prot {ours['dna+prot']:.3f}"})
if checks:
    display(pd.DataFrame(checks).set_index("claim"))

,seeds,our Spearman,our R2,their Spearman,their R2,epochs,hit the cap
variant,,,,,,,
dna,1,0.199,0.015,0.43,0.13,154,
rna,1,0.452,0.205,0.61,0.36,359,
prot,1,0.160,0.009,0.46,0.20,65,
dna+prot,1,0.439,0.182,0.52,0.28,32,
dna+rna,1,0.507,0.243,0.64,0.39,40,
dna+rna+prot,1,0.499,0.231,0.65,0.43,26,


,result,detail
claim,,
RNA is the strongest single modality,ok,"dna 0.199, rna 0.452, prot 0.160"
all three modalities beat the best single one,ok,three 0.499 against best single 0.452
DNA with RNA beats DNA with protein,ok,dna+rna 0.507 against dna+prot 0.439


## Deviations recorded for this step

| # | deviation | why |
| --- | --- | --- |
| 1 | **Encoders frozen, where IsoFormer fine-tunes them end to end** | frozen fusion is the project's premise; the paper calls encoder specialization during fine-tuning "key", so this is the largest single reason our absolute numbers sit below theirs, and measuring that distance is the point |
| 2 | Nucleotide Transformer v2 **100M** (512 wide) where IsoFormer uses the 250M (768 wide) | it is the checkpoint Stage 1 already embeds with, and keeps this comparable to step 3 |
| 3 | 1,000 training rows against their full GTEx set | the pilot is a sample; this is why the ordering rather than the absolute values is the target |
| 4 | One cross-attention layer | the paper says the module "can be also stacked" but does not say how many layers it used |
| 5 | A layer norm after each residual | the paper specifies cross-attention with residual connections and does not say what normalizes |
| 6 | Each modality attends to the others in a fixed order | "successively" does not fix an order, and residual updates do not commute |
| 7 | Each modality pooled separately, then concatenated before the linear head | the paper pools globally over length and does not say how modalities of different widths meet |
| 10 | Mean squared error as the loss | not stated; it is the obvious choice for a continuous panel |
| 9 | Batches formed under an attention-cell budget, at most 16 rows, where IsoFormer uses a fixed 64 | a row costs between 10 and 81 million attention cells here, and a fixed batch pays its longest member's cost for every row; a random 16 reached 24 GB. Grouping by length holds the worst batch near 1.2 GB. Batch composition therefore varies with length, which a fixed batch would not |
| 8 | Enformer not tried as the DNA encoder | their Table 3 gets 0.53 R2 with it against 0.43 with NT, so this is the largest single thing left on the table |

Enformer is worth a sentence to Mina. It is the one encoder swap the IsoFormer authors
measured, it is the biggest effect in their paper, and it is not in our Stage 1 set of twelve.

## What Stage 4 inherits

A second fusion architecture in the same codebase as the first, trained by the same loop,
scored by the same bootstrap, and built on a dataset whose splits are clean. That gives the
alignment question a direct test: BioLangFusion aligns at the codon and IsoFormer does not,
and after step 3 we know what a broken alignment costs. Running both over the same encoder
swaps, with the same error bars, is the experiment this project exists to run.

The open items from this notebook: whether a LucaOne stability baseline is worth running at
all, and whether Enformer can be added as a DNA encoder.